In [1]:
!pip install datasets transformers


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: C:\Users\hp\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [2]:
from datasets import load_dataset

dataset = load_dataset("kmfoda/booksum")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['bid', 'is_aggregate', 'source', 'chapter_path', 'summary_path', 'book_id', 'summary_id', 'content', 'summary', 'chapter', 'chapter_length', 'summary_name', 'summary_url', 'summary_text', 'summary_analysis', 'summary_length', 'analysis_length'],
        num_rows: 9600
    })
    validation: Dataset({
        features: ['bid', 'is_aggregate', 'source', 'chapter_path', 'summary_path', 'book_id', 'summary_id', 'content', 'summary', 'chapter', 'chapter_length', 'summary_name', 'summary_url', 'summary_text', 'summary_analysis', 'summary_length', 'analysis_length'],
        num_rows: 1484
    })
    test: Dataset({
        features: ['bid', 'is_aggregate', 'source', 'chapter_path', 'summary_path', 'book_id', 'summary_id', 'content', 'summary', 'chapter', 'chapter_length', 'summary_name', 'summary_url', 'summary_text', 'summary_analysis', 'summary_length', 'analysis_length'],
        num_rows: 1431
    })
})


In [3]:
# Voir le premier livre
exemple = dataset["train"][0]
print("Texte du livre :")
print(exemple["chapter"][:500])  # les 500 premiers caractères
print("\nRésumé de référence :")
print(exemple["summary_text"][:300])

Texte du livre :

  "Mine ear is open, and my heart prepared:
  The worst is worldly loss thou canst unfold:
  Say, is my kingdom lost?"

  SHAKESPEARE.


It was a feature peculiar to the colonial wars of North America, that
the toils and dangers of the wilderness were to be encountered before
the adverse hosts could meet. A wide and apparently an impervious
boundary of forests severed the possessions of the hostile provinces of
France and England. The hardy colonist, and the trained European who
fought at his s

Résumé de référence :
Before any characters appear, the time and geography are made clear. Though it is the last war that England and France waged for a country that neither would retain, the wilderness between the forces still has to be overcome first. Thus it is in 1757, in the New York area between the head waters of 


In [4]:
print("Nombre d'exemples dans train :", len(dataset["train"]))
print("Nombre d'exemples dans test  :", len(dataset["test"]))
print("Colonnes disponibles :", dataset["train"].column_names)

Nombre d'exemples dans train : 9600
Nombre d'exemples dans test  : 1431
Colonnes disponibles : ['bid', 'is_aggregate', 'source', 'chapter_path', 'summary_path', 'book_id', 'summary_id', 'content', 'summary', 'chapter', 'chapter_length', 'summary_name', 'summary_url', 'summary_text', 'summary_analysis', 'summary_length', 'analysis_length']


In [5]:
import re

def nettoyer_texte(texte):
    # Enlever les espaces en trop
    texte = texte.strip()
    # Enlever les caractères spéciaux inutiles
    texte = re.sub(r'\s+', ' ', texte)
    # Enlever les caractères non-ASCII
    texte = re.sub(r'[^\x00-\x7F]+', '', texte)
    return texte

# Tester sur le premier exemple
texte_brut = dataset["train"][0]["chapter"]
texte_propre = nettoyer_texte(texte_brut)
print("Avant nettoyage :", len(texte_brut), "caractères")
print("Après nettoyage :", len(texte_propre), "caractères")
print("\nAperçu :", texte_propre[:300])

Avant nettoyage : 40844 caractères
Après nettoyage : 40730 caractères

Aperçu : "Mine ear is open, and my heart prepared: The worst is worldly loss thou canst unfold: Say, is my kingdom lost?" SHAKESPEARE. It was a feature peculiar to the colonial wars of North America, that the toils and dangers of the wilderness were to be encountered before the adverse hosts could meet. A wi


In [6]:
# REMPLACER par ceci
def decouper_en_chunks(texte, taille=512, overlap=50):
    mots = texte.split()
    chunks = []
    i = 0
    while i < len(mots):
        chunk = mots[i : i + taille]
        chunks.append(' '.join(chunk))
        i += taille - overlap
    return chunks
# Tester
chunks = decouper_en_chunks(texte_propre)
print("Nombre de chunks :", len(chunks))
print("\nPremier chunk :")
print(chunks[0])

Nombre de chunks : 16

Premier chunk :
"Mine ear is open, and my heart prepared: The worst is worldly loss thou canst unfold: Say, is my kingdom lost?" SHAKESPEARE. It was a feature peculiar to the colonial wars of North America, that the toils and dangers of the wilderness were to be encountered before the adverse hosts could meet. A wide and apparently an impervious boundary of forests severed the possessions of the hostile provinces of France and England. The hardy colonist, and the trained European who fought at his side, frequently expended months in struggling against the rapids of the streams, or in effecting the rugged passes of the mountains, in quest of an opportunity to exhibit their courage in a more martial conflict. But, emulating the patience and self-denial of the practised native warriors, they learned to overcome every difficulty; and it would seem that, in time, there was no recess of the woods so dark, nor any secret place so lovely, that it might claim exemption fr

In [ ]:
def preparer_dataset(dataset, split="train", max_exemples=500):
    donnees = []
    for i in range(min(max_exemples, len(dataset[split]))):
        exemple = dataset[split][i]
        texte = nettoyer_texte(exemple["chapter"])
        resume = nettoyer_texte(exemple["summary_text"])
        chunks = decouper_en_chunks(texte)
        donnees.append({
            "chunks": chunks,
            "resume": resume,
            "nb_chunks": len(chunks)
        })
    return donnees

donnees_propres = preparer_dataset(dataset)
print(f"Dataset préparé : {len(donnees_propres)} exemples")
print(f"Exemple 1 : {donnees_propres[0]['nb_chunks']} chunks")

Dataset préparé : 100 exemples
Exemple 1 : 16 chunks


In [8]:
import json

with open("donnees_propres.json", "w") as f:
    json.dump(donnees_propres, f)

print("✅ Données sauvegardées dans donnees_propres.json")

✅ Données sauvegardées dans donnees_propres.json
